# Deep fake classification with transfer learning: EfficientNet base

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Imports

In [16]:
#Graphics
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

#Keras
from keras.models import Model
from keras import Input, layers, optimizers, callbacks

from keras.utils import image_dataset_from_directory

#Pretrained model for transfer learning
from keras.applications.efficientnet import EfficientNetB3, preprocess_input

#Saving models
import joblib

## Location of the data

In [17]:
#Lightweight dataset
train_data_dir = "/content/drive/MyDrive/dfake_data_light/train/"
val_data_dir = "/content/drive/MyDrive/dfake_data_light/valid/"
test_data_dir = "/content/drive/MyDrive/dfake_data_light/test/"

## Parameters

In [18]:
BATCH_SIZE = 128
IMAGE_SIZE = (256, 256)
IMAGE_HEIGHT = IMAGE_SIZE[0]
IMAGE_WIDTH = IMAGE_SIZE[1]
NUM_CHANNELS = 3
SEED = 42
LEARNING_RATE = 0.001

## Creating datasets from directories

In [19]:
train_ds = image_dataset_from_directory(
    train_data_dir,
    labels="inferred",
    label_mode="binary",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE)


val_ds = image_dataset_from_directory(
    val_data_dir,
    labels="inferred",
    label_mode="binary",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE)

test_ds = image_dataset_from_directory(
    test_data_dir,
    labels="inferred",
    label_mode="binary",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE)

Found 10000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


In [20]:
class_names = train_ds.class_names
print(class_names)

['fake', 'real']


In [21]:
def initialize_model(base_model):

    ######################
    ###  Architecture  ###
    ######################

    #Freezing weights of pretrained model
    base_model.trainable = False

    #Input
    inputs = Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS))

    # x = data_augmentation(inputs)
    x = preprocess_input(inputs) #Preprocessing layer specifically designed for the pretrained model
    x = base_model(x) #Transfer learning model


    x = layers.Flatten()(x)

    #Dense layers
    x = layers.Dense(128, activation='relu')(x)
    # x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    # x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)

    ################
    ##  Compiler  ##
    ################
    adam = optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(loss='binary_crossentropy',
              optimizer=adam,
              metrics=['accuracy', 'recall', 'precision'])

    return model

In [22]:
base_model = EfficientNetB3(weights="imagenet", include_top=False, input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS))

In [23]:
model = initialize_model(base_model)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 8, 8, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 98304)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │    12,583,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,374,896 (89.17 MB)

 Trainable params: 12,591,361 (48.03 MB)

 Non-trainable params: 10,783,535 (41.14 MB)

In [24]:
MODEL = 'dfake_efficientnet.keras'

modelCheckpoint = callbacks.ModelCheckpoint(MODEL,
                                            monitor="val_loss",
                                            verbose=0,
                                            save_best_only=True)

LRreducer = callbacks.ReduceLROnPlateau(monitor="val_loss",
                                        factor=0.1,
                                        patience=3,
                                        verbose=1,
                                        min_lr=0)

EarlyStopper = callbacks.EarlyStopping(monitor='val_loss',
                                       patience=3,
                                       verbose=0,
                                       restore_best_weights=True)

In [25]:
%%time
history = model.fit(train_ds,
                    epochs=10,
                    validation_data=val_ds,
                    callbacks=[modelCheckpoint, LRreducer, EarlyStopper],
                    verbose=1)

Epoch 1/10
 1/79 ━━━━━━━━━━━━━━━━━━━━ 1:56:00 89s/step - accuracy: 0.4219 - loss: 0.7685 - precision: 0.4444 - recall: 0.7742

KeyboardInterrupt: 

In [26]:
def plot_history(history):
    fig, ax = plt.subplots(1, 2, figsize=(15,5))
    ax[0].set_title('loss')
    ax[0].plot(history.epoch, history.history["loss"], label="Train loss")
    ax[0].plot(history.epoch, history.history["val_loss"], label="Validation loss")
    ax[1].set_title('accuracy')
    ax[1].plot(history.epoch, history.history["accuracy"], label="Train acc")
    ax[1].plot(history.epoch, history.history["val_accuracy"], label="Validation acc")
    ax[0].legend()
    ax[1].legend()

In [27]:
plot_history(history)

NameError: name 'history' is not defined

In [ ]:
model.evaluate(test_ds)